---
jupyter:
  jupytext:
    text_representation:
      extension: .py
      format_name: light
      format_version: '1.5'
      jupytext_version: 1.17.2
  kernelspec:
    display_name: Python 3
    language: python
    name: python3
---

# BioDYM Material Flow Analysis - Scientific Notebook

A streamlined notebook for Material Flow Analysis using the BioDYM framework with enhanced plotting capabilities.

## Workflow Overview

This notebook follows a structured approach to Material Flow Analysis:

1. **Setup and Data Loading** - Prepare environment and load input data
2. **Calculation & Validation** - Execute MFA analysis and verify results
3. **Visualization** - Comprehensive analysis and exploration
4. **Export** - Save results and generate documentation

---

# 1. Setup and Data Loading

This section prepares the analysis environment and loads the input data.

## 1.1 Environment Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown

In [ ]:
# Add BioDYM modules to path
src_path = os.path.join(os.getcwd(), 'src')
sys.path.insert(0, src_path)

In [ ]:
# Add ODYM framework to path
biodym_mfa_tool_dir = os.getcwd()
odym_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "ODYM-master_20241127", "odym", "modules"
)
sys.path.insert(0, odym_path)

In [ ]:
# Add bioDYM add-on to path
biodym_addon_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "bioDYM_add-on", "modules"
)
sys.path.insert(0, biodym_addon_path)

In [ ]:
# Import BioDYM modules
try:
    import config
    import data_loader
    import system_setup
    import utils
    from engine import solver
    from src import plotting
    import ODYM_Classes as msc
    print("✅ BioDYM modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("   Current Python path:")
    for i, path in enumerate(sys.path[:5]):  # Show first 5 paths
        print(f"   {i}: {path}")
    raise

In [ ]:
# Set up plotting
plt.style.use('default')
print("📊 Plotting environment ready")

## 1.2 Data Input Configuration

**Change this variable to your Excel file:**

In [ ]:
# Use the provided Excel file
input_file = "data/01_input/250813_CS1_simple_V1.xlsx""

In [ ]:
print(f"📁 Input file: {input_file}")

In [ ]:
# Check if input file exists
if not os.path.exists(input_file):
    print(f"⚠️ Input file not found: {input_file}")
    print("   Please update the 'input_file' variable to point to your Excel file")
    print("   Example: input_file = 'path/to/your/data.xlsx'")
    raise FileNotFoundError(f"Input file not found: {input_file}")

## 1.3 Data Loading and Validation

In [ ]:
print("\n" + "="*60)
print("[DATA] LOADING AND VALIDATING DATA")
print("="*60)

In [ ]:
# Load Excel file
try:
    input_data = pd.read_excel(
        input_file,
        sheet_name=None,
        header=0,
        engine='openpyxl',
        na_values=['N.A.', 'NA', 'n/a']
    )
    print(f"✅ Excel file loaded: {len(input_data)} sheets")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    raise

In [ ]:
# Display sheet overview
print("\n📋 Sheet Overview:")
for sheet_name, df in input_data.items():
    print(f"   {sheet_name}: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Validate required sheets
required_sheets = [
    '1_1_Definition_Flows',
    '1_2_Data_Flows', 
    '2_1_Definition_Processes',
    '2_4_Initial_Stock',  # Correct sheet name
    '2_5_dynamic_tcs'
]

In [ ]:
missing_sheets = [sheet for sheet in required_sheets if sheet not in input_data.keys()]
if missing_sheets:
    print(f"\n⚠️ Missing required sheets: {missing_sheets}")
else:
    print("\n✅ All required sheets present")

## 1.4 System Configuration Extraction

In [ ]:
print("\n" + "="*60)
print("⚙️ EXTRACTING CONFIGURATION")
print("="*60)

In [ ]:
# Extract time range from flow data
flow_data = input_data['1_2_Data_Flows']
years = sorted(flow_data['Year_Flow'].unique())
start_year = int(min(years))
end_year = int(max(years))

In [ ]:
print(f"📅 Time range: {start_year} - {end_year}")

In [ ]:
# Extract elements from flow data
elements = ['material', 'WC', 'DM', 'CC']  # Default elements
print(f"🧪 Elements: {elements}")

In [ ]:
# Check for Monte Carlo parameters
has_mc = '4_1_Uncertainty_Parameters' in input_data.keys()
print(f"🎲 Monte Carlo available: {'Yes' if has_mc else 'No'}")

In [ ]:
# Check for DSM parameters
has_dsm = '3_1_Definition_DSM' in input_data.keys()
print(f"📈 DSM available: {'Yes' if has_dsm else 'No'}")

In [ ]:
# Check for FOMP parameters
has_fomp = '3_2_Definition_FOMP' in input_data.keys()
print(f"🌱 FOMP available: {'Yes' if has_fomp else 'No'}")

## 1.5 Configuration Review

In [ ]:
print("\n" + "="*60)
print("✅ CONFIGURATION CONFIRMATION")
print("="*60)

In [ ]:
config_summary = f"""
**Analysis Configuration:**
- Input File: {input_file}
- Time Range: {start_year} - {end_year}
- Elements: {', '.join(elements)}
- Monte Carlo: {'Enabled' if has_mc else 'Disabled'}
- DSM: {'Enabled' if has_dsm else 'Disabled'}
- FOMP: {'Enabled' if has_fomp else 'Disabled'}
"""

In [ ]:
display(Markdown(config_summary))

---
BioDYM Extension Notice
---

In [ ]:
from IPython.display import display, Markdown

In [ ]:
display(Markdown('''
**Note:** The stock-outflow transfer coefficient feature is a custom extension to the ODYM framework, developed specifically for BioDYM. It is not part of the standard ODYM release.
'''))

# 2. Calculation & Validation

This section executes the MFA calculation and immediately validates the results through mass balance checks.

## 2.1 Model Initialization

In [ ]:
print("\n" + "="*60)
print("🚀 RUNNING MFA CALCULATION")
print("="*60)

In [ ]:
# 1. Setup model scope
print("📋 Setting up model scope...")
try:
    model_classification, index_table = system_setup.define_model_scope(
        start_year, end_year, elements
    )
    print("✅ Model scope defined")
except Exception as e:
    print(f"❌ Error setting up model scope: {e}")
    raise

In [ ]:
# 2. Initialize MFA system
print("🔧 Initializing MFA system...")
try:
    mfa_system_base = system_setup.initialize_mfa_system(
        model_classification, index_table
    )
    print("✅ MFA system initialized")
except Exception as e:
    print(f"❌ Error initializing MFA system: {e}")
    raise

In [ ]:
# 3. Load and define processes
print("📊 Loading processes and data...")
try:
    mfa_system_base, all_excel_data = system_setup.load_and_define_processes(
        mfa_system_base, input_file, data_loader
    )
    print("✅ Processes and data loaded")
except Exception as e:
    print(f"❌ Error loading processes: {e}")
    raise

In [ ]:
# 4. Load parameters
print("⚙️ Loading parameters...")
try:
    dsm_params = data_loader.load_dsm_parameters(all_excel_data)
    fomp_params = data_loader.load_fomp_parameters(all_excel_data)
    uncertainty_params = data_loader.load_uncertainty_definitions(all_excel_data)
    print("✅ Parameters loaded")
except Exception as e:
    print(f"❌ Error loading parameters: {e}")
    raise

## 2.2 MFA Calculation Execution

In [ ]:
# 5. Define flows and parameters
print("🔗 Defining flows and parameters...")
try:
    mfa_system_configured, _ = system_setup.define_flows_and_parameters(
        mfa_system_base, all_excel_data
    )
    print(f"✅ System configured: {len(mfa_system_configured.ProcessList)} processes, "
          f"{len(mfa_system_configured.FlowDict)} flows, {len(mfa_system_configured.StockDict)} stocks")
except Exception as e:
    print(f"❌ Error defining flows and parameters: {e}")
    raise

In [ ]:
# 5.1 Process dynamic TCs
print("🔄 Processing dynamic transfer coefficients...")
try:
    dynamic_tc_sheet = all_excel_data.get('2_5_dynamic_tcs')
    if dynamic_tc_sheet is not None and not dynamic_tc_sheet.empty:
        dynamic_tcs = system_setup.create_dynamic_tc_parameters(
            dynamic_tc_sheet, mfa_system_configured.IndexTable.Classification['Time'].Items
        )
        # Add dynamic TCs to the system parameters
        for name, values in dynamic_tcs.items():
            mfa_system_configured.ParameterDict[name] = msc.Parameter(
                Name=name,
                ID=len(mfa_system_configured.ParameterDict) + 1,
                Values=values,
                Unit="1"
            )
        print(f"✅ Dynamic TCs processed: {len(dynamic_tcs)} parameters added")
    else:
        print("ℹ️ No dynamic TCs found in input data")
except Exception as e:
    print(f"⚠️ Warning: Could not process dynamic TCs: {e}")
    print("   Continuing with static TCs only")

In [ ]:
# 6. Run calculation
print("🧮 Running calculation...")
try:
    mfa_system_with_results, dsm_details = solver.run_mfa_calculation(
        mfa_system_configured, dsm_params, fomp_params, config
    )
    print("✅ Calculation completed successfully!")
except Exception as e:
    print(f"❌ Calculation error: {e}")
    import traceback
    traceback.print_exc()
    raise

## 2.3 Mass Balance Validation

In [ ]:
print("\n" + "="*60)
print("⚖️ MASS BALANCE VERIFICATION")
print("="*60)

In [ ]:
# Calculate mass balance errors
mass_balance_errors = []
for process in mfa_system_with_results.ProcessList:
    if hasattr(process, 'MassBalance') and process.MassBalance is not None:
        for year_idx, year in enumerate(range(start_year, end_year + 1)):
            for element_idx, element in enumerate(elements):
                error = process.MassBalance[year_idx, element_idx]
                if abs(error) > 1e-6:  # Significant error threshold
                    mass_balance_errors.append({
                        'Process': process.Name,
                        'Year': year,
                        'Element': element,
                        'Error': error
                    })

In [ ]:
if mass_balance_errors:
    print("⚠️ Mass balance errors detected:")
    error_df = pd.DataFrame(mass_balance_errors)
    display(error_df)
else:
    print("✅ All mass balances within acceptable limits")

In [ ]:
# Mass Balance Error Visualization
print("\n⚖️ Creating mass balance error plots...")
try:
    # Display the general overview plot
    print("\n--- General Overview: Total Absolute Errors ---")
    
    # --- DEBUG CELL ---
    print("\n" + "="*30)
    print("🔬 DEBUGGING FLOW F_00_02")
    print("="*30)
    try:
        debug_flow = mfa_system_with_results.FlowDict.get('F_00_02')
        if debug_flow:
            print("Elements:", mfa_system_with_results.Elements)
            print("Flow F_00_02 Values (first 5 years):")
            print(pd.DataFrame(debug_flow.Values[:5, :], columns=mfa_system_with_results.Elements).to_string())
        else:
            print("Flow F_00_02 not found in the system.")
    except Exception as e:
        print(f"Error during debug: {e}")
    print("="*30)
    # --- END DEBUG CELL ---
    
    plotting.plot_total_mass_balance_error(mfa_system_with_results)
    
    # Display the time-specific interactive plot
    print("\n--- Time-Specific View: Interactive Error Check ---")
    plotting.plot_optimized_mass_balance_error(mfa_system_with_results)
    
    print("✅ Mass balance error plots created")
except Exception as e:
    print(f"⚠️ Could not create mass balance error plots: {e}")

In [ ]:
# === Sankey-Style Block Flow Diagram ===
from src.plotting.graphviz_flow_charts import plot_graphviz_flow_chart_sankey_style
from IPython.display import Image, display
import tempfile
from datetime import datetime

In [ ]:
print("\n" + "="*60)
print("🟦 SANKEY-STYLE BLOCK FLOW DIAGRAM")
print("="*60)

In [ ]:
# Generate Sankey-Style Block Flow Diagram
try:
    dot_sankey = plot_graphviz_flow_chart_sankey_style(
        input_file,
        title="BioDYM System - Sankey-Style Block Flow Diagram",
        rankdir="LR",
        ranksep=1.0,
        nodesep=0.5
    )
    if dot_sankey is not None:
        # Create temporary file with better handling
        import os
        tmp_dir = tempfile.gettempdir()
        tmp_filename = f"biodym_sankey_style_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        tmp_path = os.path.join(tmp_dir, tmp_filename)
        
        # Render the diagram
        dot_sankey.render(tmp_path, format='png', cleanup=True)
        png_path = tmp_path + '.png'
        
        if os.path.exists(png_path):
            print("✅ Sankey-style block flow diagram created")
            display(Image(filename=png_path))
            # Clean up the file after display
            try:
                os.remove(png_path)
            except:
                pass  # Ignore cleanup errors
        else:
            print("❌ Failed to create PNG file")
    else:
        print("❌ Failed to create Sankey-style block flow diagram")
except Exception as e:
    print(f"❌ Error creating Sankey-style block flow diagram: {e}")

## 2.4 Results Overview

In [ ]:
print("\n" + "="*60)
print("📈 RESULTS OVERVIEW")
print("="*60)

In [ ]:
# Display final stock values
print("\n📊 Final Stock Values (Year {end_year}):")
final_stocks = []
for stock_name, stock in mfa_system_with_results.StockDict.items():
    if stock_name.startswith('S_'):  # Absolute stocks only
        final_value = stock.Values[-1, 0]  # Material dimension, final year
        final_stocks.append({
            'Stock': stock_name,
            'Final Value (Mg)': final_value
        })

In [ ]:
if final_stocks:
    stocks_df = pd.DataFrame(final_stocks)
    display(stocks_df)

In [ ]:
# Display flow summary
print("\n🔄 Flow Summary:")
flow_summary = []
for flow_id, flow in mfa_system_with_results.FlowDict.items():
    avg_flow = np.mean(flow.Values[:, 0])  # Average material flow
    flow_summary.append({
        'Flow ID': flow_id,
        'From': flow.P_Start,
        'To': flow.P_End,
        'Avg Flow (Mg/year)': avg_flow
    })

In [ ]:
if flow_summary:
    flows_df = pd.DataFrame(flow_summary)
    display(flows_df.head(10))  # Show first 10 flows

# 3. Visualization

This section provides comprehensive analysis and exploration through various visualization tools.

In [ ]:
print("\n" + "="*60)
print("📊 VISUALIZATION")
print("="*60)

## 3.1 System Overview

In [ ]:
print("\n" + "-"*40)
print("3.1 SYSTEM OVERVIEW")
print("-"*40)

### 3.1.1 Material Flow Sankey Diagram

In [ ]:
print("🔗 Creating interactive Sankey diagram...")
try:
    # Use the enhanced interactive Sankey function with DSM/FOMP parameters
    plotting.plot_interactive_sankey(mfa_system_with_results, dsm_params, fomp_params)
    print("✅ Interactive Sankey diagram created")
    print("   📊 Features: Multi-process selection, color coding, export options")
    print("   🎨 Process types: Regular (blue), DSM (orange), FOMP (green)")
    print("   📁 Export: PNG with timestamped filenames in organized folders")
except Exception as e:
    print(f"⚠️ Could not create interactive Sankey diagram: {e}")
    import traceback
    traceback.print_exc()

### 3.1.2 Process Dynamics Analysis

In [ ]:
print("\n📊 Creating process dynamics analysis...")
try:
    # Use the process dynamics function for general process analysis
    if '2_1_Definition_Processes' in input_data:
        process_definitions = input_data['2_1_Definition_Processes']
        plotting.plot_process_dynamics(mfa_system_with_results, process_definitions)
        print("✅ Process dynamics analysis created")
        print("   📊 Features: Inflow, Stock, and Outflow for each process")
        print("   📈 Interactive: Process and element selection")
        print("   🎨 Smart titles based on process types")
    else:
        print("ℹ️ Process definitions not available for dynamics analysis")
except Exception as e:
    print(f"⚠️ Could not create process dynamics analysis: {e}")

### 3.1.3 Stock Levels Bar Chart

In [ ]:
print("\n📊 Creating stock levels bar chart...")
try:
    # Use the new stock bar chart function
    plotting.plot_stock_bar_chart(mfa_system_with_results, title="Stock Levels Over Time")
    print("✅ Stock levels bar chart created")
    print("   📊 Features: Interactive slider to view stocks per year")
    print("   🎨 Styling: Clear, publication-ready design")
except Exception as e:
    print(f"⚠️ Could not create stock levels bar chart: {e}")

## 3.2 Individual Process Analysis

In [ ]:
print("\n" + "-"*40)
print("3.2 INDIVIDUAL PROCESS ANALYSIS")
print("-"*40)

### 3.2.1 DSM Process Analysis

In [ ]:
print("\n📈 3.2.1 DSM Process Analysis:")
try:
    if has_dsm and dsm_details:
        plotting.plot_dsm_stock_details(mfa_system_with_results, dsm_params, dsm_details)
        print("✅ DSM process analysis plots created")
        print("   📊 Features: Individual/Cumulative views, lifetime display")
        print("   🎨 Enhanced styling with export functionality")
    else:
        print("ℹ️ No DSM processes available")
except Exception as e:
    print(f"⚠️ Could not create DSM process analysis: {e}")

### 3.2.2 DSM Stock Composition Analysis

In [ ]:
print("\n📊 3.2.2 DSM Stock Composition Analysis:")
try:
    if has_dsm and dsm_details:
        plotting.plot_dynamic_stock_composition(dsm_details, mfa_system_with_results)
        print("✅ DSM stock composition analysis created")
        print("   📊 Features: Initial stock decay vs new stock accumulation")
        print("   📈 Options: Line charts or stacked bar charts")
        print("   🎨 Interactive: Process and element selection")
    else:
        print("ℹ️ No DSM processes available")
except Exception as e:
    print(f"⚠️ Could not create DSM stock composition analysis: {e}")

### 3.2.3 FOMP Process Analysis

In [ ]:
print("\n🌱 3.2.3 FOMP Process Analysis:")
try:
    if has_fomp and fomp_params:
        plotting.plot_fomp_stock_details(mfa_system_with_results, fomp_params)
        print("✅ FOMP process analysis plots created")
    else:
        print("ℹ️ No FOMP processes available")
except Exception as e:
    print(f"⚠️ Could not create FOMP process analysis: {e}")

## 3.3 Detailed Component Analysis

In [ ]:
print("\n" + "-"*40)
print("3.3 DETAILED COMPONENT ANALYSIS")
print("-"*40)

### 3.3.1 Individual Flow Analysis

In [ ]:
print("\n🔄 Creating individual flow analysis...")
try:
    plotting.plot_flow_dynamics(mfa_system_with_results)
    print("✅ Individual flow analysis created")
    print("   📊 Features: Multi-flow selection, cumulative vs. individual values")
    print("   📈 Options: Bar/line charts, element-specific analysis")
except Exception as e:
    print(f"⚠️ Could not create individual flow analysis: {e}")

Note: Additional flow chart visualizations removed - keeping only Sankey-style block flow diagram

# 4. Export

This section saves results and generates documentation for the analysis.

In [ ]:
print("\n" + "="*60)
print("💾 EXPORTING RESULTS")
print("="*60)

## 4.1 Results Export

In [ ]:
# Export to Excel
output_file = "data/02_output/results_scientific.xlsx"
try:
    utils.export_results_to_excel(mfa_system_with_results, output_file)
    print(f"✅ Results exported to: {output_file}")
except Exception as e:
    print(f"⚠️ Export error: {e}")

## 4.2 Configuration Export

In [ ]:
# Export configuration summary
config_file = output_file.replace('.xlsx', '_config.xlsx')
try:
    config_summary = pd.DataFrame([{
        'Input File': input_file,
        'Start Year': start_year,
        'End Year': end_year,
        'Elements': ', '.join(elements),
        'Monte Carlo': has_mc,
        'DSM': has_dsm,
        'FOMP': has_fomp
    }])
    config_summary.to_excel(config_file, index=False)
    print(f"✅ Configuration exported to: {config_file}")
except Exception as e:
    print(f"⚠️ Config export error: {e}")

## 4.3 Analysis Summary

In [ ]:
print("\n" + "="*60)
print("🎉 ANALYSIS COMPLETE")
print("="*60)

In [ ]:
summary = f"""
**Analysis Summary:**
- ✅ Input file processed successfully
- ✅ Configuration extracted automatically
- ✅ MFA calculation completed
- ✅ Mass balance verified
- ✅ Visualizations generated
- ✅ Results exported

**Key Results:**
- Time period: {start_year} - {end_year}
- Processes analyzed: {len(mfa_system_with_results.ProcessList)}
- Flows tracked: {len(mfa_system_with_results.FlowDict)}
- Stocks modeled: {len(mfa_system_with_results.StockDict)}
- Mass balance errors: {len(mass_balance_errors)}

**Files Generated:**
- Main results: {output_file}
- Configuration: {config_file}
"""

In [ ]:
display(Markdown(summary))

In [ ]:
print("\n📊 Analysis completed successfully!")

In [ ]:
# Monte Carlo Analysis with proper error handling
try:
    from engine.mc_simulation import run_mc_simulation
    from plotting.mc_visuals import plot_interactive_mc_histogram, plot_interactive_tornado

    print("\n" + "="*60)
    print("🎲 MONTE CARLO SIMULATION")
    print("="*60)

    if has_mc:
        mc_results = run_mc_simulation(
            mfa_system_configured, input_data, dsm_params, fomp_params, config
        )

        if mc_results is not None and not mc_results.empty:
            print("✅ Monte Carlo simulation completed")
            print("📊 Creating Monte Carlo stock histogram...")
            plot_interactive_mc_histogram(mc_results)
            stock_mc_cols = [col for col in mc_results.columns if col.endswith('_mc') and col.startswith('S_')]
            if stock_mc_cols:
                print("📈 Creating tornado plot for sensitivity analysis...")
                plot_interactive_tornado(mc_results)
            else:
                print("ℹ️ No stock columns found for tornado plot.")
        else:
            print("ℹ️ Monte Carlo simulation ran, but no results were generated. Check uncertainty definitions.")
    else:
        print("ℹ️ Monte Carlo analysis is disabled in the configuration. Skipping.")
except ImportError as e:
    print(f"⚠️ Monte Carlo modules not available: {e}")
except Exception as e:
    print(f"⚠️ Monte Carlo simulation failed: {e}")
    import traceback
    traceback.print_exc()